## **03 - Scraping Booking.com (Playwright)**


### **Pourquoi Playwright (et pas `requests` + `bs4`, ni `scrapy`)?**

Booking charge une partie du contenu en JavaScript et applique des mecanismes anti-bot.

- `requests` + `BeautifulSoup`: rapide, mais fragile si le HTML initial ne contient pas toutes les donnees rendues par JS.
- `scrapy`: excellent pour du crawl a grande echelle, mais ici il faut d'abord fiabiliser le rendu navigateur et les interactions.
- `playwright`: pilote un vrai navigateur (Chromium), rend le JavaScript, permet d'attendre les bons selecteurs et de gerer les interactions.

Conclusion: pour un scraping cible et robuste sur Booking, Playwright est le meilleur compromis.

> Note legale: respecte toujours les CGU du site cible, limite la frequence des requetes et evite toute surcharge.

In [1]:
import re
import asyncio
import pandas as pd

from datetime import date, timedelta
from pathlib import Path
from urllib.parse import quote_plus
from playwright.async_api import async_playwright, TimeoutError as PlaywrightTimeoutError

In [2]:
# Parametres de scraping
MAX_HOTELS_PER_CITY = 5
HEADLESS = True
DEBUG_SCRAPE = True
DEFAULT_COUNTRY = "France"
CHECKIN_DATE = (date.today() + timedelta(days=10)).isoformat()
CHECKOUT_DATE = (date.today() + timedelta(days=17)).isoformat()

scored_path = Path("../outputs/data/cities_weather_scored.csv")
if not scored_path.exists():
    raise FileNotFoundError(
        "Fichier introuvable: ../outputs/data/cities_weather_scored.csv. Execute d'abord le notebook 02."
    )

scored_df = pd.read_csv(scored_path)

if "city_name" not in scored_df.columns:
    raise ValueError("La colonne 'city_name' est requise dans cities_weather_scored.csv")

if "country" not in scored_df.columns:
    scored_df["country"] = DEFAULT_COUNTRY

if "weather_rank" in scored_df.columns:
    top5_df = scored_df.nsmallest(5, "weather_rank").copy()
else:
    # Fallback: on garde les 5 premieres lignes si le rang n'est pas present
    top5_df = scored_df.head(5).copy()

top5_df[["city_name", "country"]]

,city_name,country
0,Aix en Provence,France
1,Grenoble,France
2,Paris,France
3,Besancon,France
4,Annecy,France


In [3]:
def build_booking_search_url(city: str, country: str, checkin: str, checkout: str) -> str:
    query = quote_plus(f"{city}, {country}")
    return (
        "https://www.booking.com/searchresults.fr.html"
        f"?ss={query}"
        f"&checkin={checkin}&checkout={checkout}"
        "&group_adults=2&no_rooms=1&group_children=0"
    )


def parse_lat_lon_from_html(html: str):
    atlas_match = re.search(r'data-atlas-latlng="([0-9\.-]+),([0-9\.-]+)"', html)
    if atlas_match:
        return float(atlas_match.group(1)), float(atlas_match.group(2))

    json_match = re.search(r'"latitude"\s*:\s*([0-9\.-]+).*?"longitude"\s*:\s*([0-9\.-]+)', html, re.S)
    if json_match:
        return float(json_match.group(1)), float(json_match.group(2))

    return None, None


async def extract_listing_cards(page):
    selectors = [
        'div[data-testid="property-card"]',
        'div[data-testid="property-card-container"]',
    ]
    for selector in selectors:
        cards = page.locator(selector)
        if await cards.count() > 0:
            return cards
    return page.locator('article')


In [4]:
async def scrape_city_hotels(page, city: str, country: str, max_hotels: int = 5):
    search_url = build_booking_search_url(city, country, CHECKIN_DATE, CHECKOUT_DATE)
    await page.goto(search_url, wait_until="domcontentloaded", timeout=90_000)
    await asyncio.sleep(2)

    # Accepte le bandeau cookies si present
    for cookie_selector in [
        'button#onetrust-accept-btn-handler',
        'button:has-text("J\'accepte")',
        'button:has-text("Accept")',
    ]:
        btn = page.locator(cookie_selector).first
        if await btn.count() > 0:
            try:
                await btn.click(timeout=2_000)
                break
            except Exception:
                pass

    try:
        await page.wait_for_selector('div[data-testid="property-card"]', timeout=20_000)
    except (PlaywrightTimeoutError, asyncio.CancelledError):
        pass

    page_html = await page.content()
    lower_html = page_html.lower()
    if "captcha" in lower_html or "robot" in lower_html or "verify you are human" in lower_html:
        print(f"  -> Blocage anti-bot detecte pour {city}")

    cards = await extract_listing_cards(page)
    hotels = []
    cards_count = await cards.count()

    if cards_count == 0 and DEBUG_SCRAPE:
        debug_dir = Path("../outputs/debug")
        debug_dir.mkdir(parents=True, exist_ok=True)
        safe_city = re.sub(r"[^a-zA-Z0-9_-]+", "_", city)
        screenshot_path = debug_dir / f"booking_{safe_city}.png"
        await page.screenshot(path=str(screenshot_path), full_page=True)
        print(f"  -> 0 carte trouvee. URL finale: {page.url}")
        print(f"  -> Screenshot debug: {screenshot_path.resolve()}")

    for i in range(min(cards_count, max_hotels)):
        card = cards.nth(i)

        title_locator = card.locator('div[data-testid="title"]').first
        name = await title_locator.inner_text(timeout=3_000) if await title_locator.count() else None

        link_el = card.locator('a[data-testid="title-link"]').first
        if await link_el.count() == 0:
            link_el = card.locator("a").first
        hotel_url = await link_el.get_attribute("href") if await link_el.count() else None
        if hotel_url and hotel_url.startswith("/"):
            hotel_url = f"https://www.booking.com{hotel_url}"

        score_text = None
        score_selectors = [
            'div[data-testid="review-score"]',
            'div[aria-label*="Scored"]',
            'div[aria-label*="Note"]',
        ]
        for s in score_selectors:
            score_el = card.locator(s).first
            if await score_el.count() > 0:
                score_text = await score_el.inner_text(timeout=2_000)
                break

        lat, lon, description = None, None, None
        if hotel_url:
            detail = await page.context.new_page()
            try:
                await detail.goto(hotel_url, wait_until="domcontentloaded", timeout=90_000)
                detail_html = await detail.content()
                lat, lon = parse_lat_lon_from_html(detail_html)

                desc_selectors = [
                    '#property_description_content',
                    'p[data-testid="property-description"]',
                    'div[data-testid="property-description"]',
                ]
                for ds in desc_selectors:
                    desc_el = detail.locator(ds).first
                    if await desc_el.count() > 0:
                        description = (await desc_el.inner_text(timeout=3_000)).strip()
                        break
            except Exception:
                pass
            finally:
                await detail.close()

        hotels.append(
            {
                "city_name": city,
                "country": country,
                "hotel_name": name,
                "booking_url": hotel_url,
                "latitude": lat,
                "longitude": lon,
                "score_text": score_text,
                "description": description,
            }
        )

        await asyncio.sleep(1.2)

    return hotels


In [5]:
async def run_booking_scrape(top5_df: pd.DataFrame) -> pd.DataFrame:
    all_hotels = []

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=HEADLESS, slow_mo=50)
        context = await browser.new_context(
            locale="fr-FR",
            user_agent=(
                "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
                "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
            ),
            viewport={"width": 1366, "height": 900},
        )
        page = await context.new_page()

        for row in top5_df.itertuples(index=False):
            city = row.city_name
            country = row.country
            print(f"Scraping: {city}, {country}")

            try:
                city_hotels = await scrape_city_hotels(
                    page,
                    city=city,
                    country=country,
                    max_hotels=MAX_HOTELS_PER_CITY,
                )
                all_hotels.extend(city_hotels)
                print(f"  -> {len(city_hotels)} hotels recuperes")
            except asyncio.CancelledError:
                print(f"  -> Scraping annule pour {city}")
            except Exception as e:
                print(f"  -> Echec sur {city}: {e}")

        await context.close()
        await browser.close()

    return pd.DataFrame(all_hotels)


hotels_df = await run_booking_scrape(top5_df)
hotels_df.head()

Scraping: Aix en Provence, France
  -> Blocage anti-bot detecte pour Aix en Provence
  -> 5 hotels recuperes
Scraping: Grenoble, France
  -> Blocage anti-bot detecte pour Grenoble
  -> 5 hotels recuperes
Scraping: Paris, France
  -> Blocage anti-bot detecte pour Paris
  -> 5 hotels recuperes
Scraping: Besancon, France
  -> Blocage anti-bot detecte pour Besancon
  -> 5 hotels recuperes
Scraping: Annecy, France
  -> Blocage anti-bot detecte pour Annecy
  -> 5 hotels recuperes


,city_name,country,hotel_name,booking_url,latitude,longitude,score_text,description
0,Aix en Provence,France,Appart'hôtel Odalys City - Aix en Provence Cen...,https://www.booking.com/hotel/fr/atrium-d-anai...,43.524091,5.455216,"Avec une note de 8,0\n8,0\nTrès bien\n1 538 ex...","Situé à Aix-en-Provence, l’Cours Mirabeau, App..."
1,Aix en Provence,France,Renaissance Aix-en-Provence Hotel,https://www.booking.com/hotel/fr/renaissance-a...,43.526331,5.437964,"Avec une note de 8,6\n8,6\nSuperbe\n1 412 expé...",Situé à moins de 10 minutes à pied du centre d...
2,Aix en Provence,France,Domaine de Carraire,https://www.booking.com/hotel/fr/domaine-de-ca...,43.566063,5.384448,"Avec une note de 8,5\n8,5\nTrès bien\n249 expé...",L’établissement Domaine de Carraire se trouve ...
3,Aix en Provence,France,Hôtel Birdy by Happyculture,https://www.booking.com/hotel/fr/royal-mirabea...,43.481430,5.365614,"Avec une note de 8,6\n8,6\nSuperbe\n2 490 expé...",L’Hôtel Birdy by Happyculture se trouve à 10 m...
4,Aix en Provence,France,Grand Hôtel Roi René Aix - MGallery Collection,https://www.booking.com/hotel/fr/grand-mercure...,NaN,NaN,"Avec une note de 8,4\n8,4\nTrès bien\n1 242 ex...",NaN


In [6]:
# Nettoyage leger du score numerique
hotels_df["score"] = (
    hotels_df["score_text"]
    .astype(str)
    .str.extract(r"([0-9]+[\.,]?[0-9]*)", expand=False)
    .str.replace(",", ".", regex=False)
)
hotels_df["score"] = pd.to_numeric(hotels_df["score"], errors="coerce")

hotels_df = hotels_df[
    [
        "city_name",
        "country",
        "hotel_name",
        "booking_url",
        "latitude",
        "longitude",
        "score",
        "description",
    ]
]

hotels_df.sample(min(10, len(hotels_df)))

,city_name,country,hotel_name,booking_url,latitude,longitude,score,description
5,Grenoble,France,Apparthôtel Tempologis - St Germain à Grenoble,https://www.booking.com/hotel/fr/tempologis-gr...,45.179273,5.734339,8.4,"Situé à Grenoble, le Apparthôtel Tempologis - ..."
17,Besancon,France,Hôtel Siatel Besançon Chateaufarine,https://www.booking.com/hotel/fr/siatel-chatea...,NaN,NaN,7.7,NaN
24,Annecy,France,Auberge de Jeunesse HI Annecy,https://www.booking.com/hotel/fr/auberge-de-je...,45.891422,6.133705,7.1,L’établissement Auberge de Jeunesse HI Annecy ...
12,Paris,France,Moxy Paris Bastille,https://www.booking.com/hotel/fr/moxy-paris-ba...,48.857528,2.369940,8.0,Le Moxy Paris Bastille est un hôtel situé entr...
8,Grenoble,France,Campanile Grenoble Centre Gare,https://www.booking.com/hotel/fr/campanile-gre...,45.190519,5.716019,7.4,Le Campanile Grenoble Centre Gare est situé à ...
14,Paris,France,Hotel Mercure La Sorbonne Saint-Germain-des-Prés,https://www.booking.com/hotel/fr/paris-la-sorb...,48.849413,2.343229,8.7,Cet hôtel Mercure est situé en face de l'Unive...
10,Paris,France,Magnificient 2 Bedrooms with terrace next to c...,https://www.booking.com/hotel/fr/magnificient-...,NaN,NaN,8.4,NaN
15,Besancon,France,ibis Styles Besançon,https://www.booking.com/hotel/fr/besancon.fr.h...,47.254253,6.021640,7.8,"Idéalement situé dans un parc, l'hôtel ibis St..."
23,Annecy,France,Impérial Palace,https://www.booking.com/hotel/fr/imperial-pala...,45.903747,6.144694,8.9,"Au cœur des Alpes françaises, l’Impérial Palac..."
19,Besancon,France,Temis hôtel Besançon,https://www.booking.com/hotel/fr/all-suites-be...,47.254357,5.995480,8.1,L’établissement Temis hôtel Besançon vous accu...


In [7]:
output_data_dir = Path("../outputs/data")
output_data_dir.mkdir(parents=True, exist_ok=True)

booking_output_file = output_data_dir / "top5_booking_hotels.csv"
hotels_df.to_csv(booking_output_file, index=False)

print(f"Export Booking: {booking_output_file.resolve()}")
print(f"Nombre de lignes: {len(hotels_df)}")

Export Booking: /home/raphael/DataProjects/jedha-certif/projet-kayak-bloc1/outputs/data/top5_booking_hotels.csv
Nombre de lignes: 25


## Installation (si necessaire)

```bash
pipenv install
pipenv run playwright install chromium
```

Puis execute ce notebook avec le kernel de l'environnement Pipenv.